# **1. Import required Python packages**

This cell loads a bunch of Python packages which are required to run the notebook. You should use the **cheminf.yaml** environment for running this notebook!

For example, some of these are related to the processing of data (`numpy` and `pandas`), some for actually performing the linear modelling (`sklearn`) and some for plotting the results of your modeling (`matplotlib` and `seaborn`).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from scipy import stats

from sklearn import linear_model
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

import plot_model

# **2. Load in your data**

This cell loads the data in your csv file to a pandas dataframe. Just like before, we will also make a separate features dataframe.

In [ ]:
df = pd.read_csv("data/amide_coupling.csv", index_col=0)

# as this will cause issues later, if there are any rows where the output is missing, we should drop them
df = df.dropna(subset=["ln rate"])

display(df.head())

df_features = df.copy().drop(columns=["ln rate"])
display(df_features.head())

# **3. Examine the data spread and feature correlations**

Next, we will plot a histogram to visually represent the spread of data (the spread of our `response_col`).

In [ ]:
response_col = "ln rate"

sns.histplot(data=df, x=response_col)
plt.show()

Now we will have a look and see what features within our dataset are correlated with each other. In our case we will use the Pearson corellation coefficient to generate correlation matrix.

In [ ]:
df_corr = df_features.corr()

# this masks the upper triangle of the correlation matrix, which is redundant
mask = np.triu(np.ones_like(df_corr, dtype=bool))

plt.figure(figsize=(10, 8))
sns.heatmap(df_corr, mask=mask, cmap="coolwarm", vmin=-1, vmax=1)
plt.title("Correlation Matrix")
plt.show()

# **4. Find univariate correlations**

Before going immediately to complex model architectures, it is usually sensible to start with the simplest type of model and gradually increase the complexity as needed. 

Here, we will start building some simple univariate linear regression models. The below script iterates through all of the features and attempts to find a correlation (within a certain user specified limit, `r2_cutoff`).

Set the value for `r2_cutoff` below, this keeps all models with $R^2$ greater than the cutoff.

In [ ]:
r2_cutoff = 0.5

Now we can find some models!

Again, like before, we should turn our data into numpy arrays.

In [ ]:
X = df_features.to_numpy()
X_names = df_features.columns
y = df[response_col].to_numpy()

In [ ]:
# Iterate through all features

for feature in df_features.columns:
    # first generate a linear regression model for this feature
    slope, intercept, r_value, p_value, std_err = stats.linregress(X[:, df_features.columns.get_loc(feature)], y)
    fit_line = intercept + slope * df_features[feature]
    r2 = r_value**2
    print(f"Feature: {feature}, R^2: {r2:.3f}")
    if r2 > r2_cutoff:
        print(f"  R^2 is above cutoff of {r2_cutoff}, plotting...")
        plt.scatter(df_features[feature], df[response_col], s=100, edgecolor="k")
        plt.plot(df_features[feature], fit_line, color="crimson", label=f"Fit ($R^2$={r2:.3f})")
        plt.xlabel(feature)
        plt.ylabel(response_col)
        plt.legend()
        plt.show()

# **5. MLR modeling (find models with 2 or more components)**

## **5.1 Prepare data**

Before starting the MLR portion of the notebook, the data you previously inputted needs to be prepared.

It's pointless including two descriptors in the same model that we know are highly correlated (i.e., these will likely provide similar physical information about the molecules), so it's a good idea to remove them. Out of a pair of correlated descriptors it is up to you which one you throw out - this could be done based on some prior chemistry knowledge or simply done by the computer.

In [ ]:
cutoff = 0.9

print(f"Number of features before curation: {df_features.shape[1]}")
df_corr = df_features.corr()

df_not_correlated = ~(df_corr.mask(np.tril(np.ones([len(df_corr)]*2, dtype=bool))).abs() > cutoff).any()
un_corr_idx = df_not_correlated.loc[df_not_correlated[df_not_correlated.index] == True].index
df_features = df_features.copy()[un_corr_idx]
print(f"Number of features after curation: {df_features.shape[1]}")

In [ ]:
X = df_features.values
y = df[response_col].values

Next, we need to (1) define a training/test split and (2) scale the features.

- The **training set** is the portion of the data which the model will be intially fit (or trained) on.
- The **test set** is the portion of the data which will be kept to one side and be used to later validate the model (this is also sometimes referred to as the **validation set**).

The train/test split can be determined in a number of ways. First you will determine the ratio of train and test data points. After that, you will determine the method of splitting the data:

- **Random** selects the point randomly
- **y-Equidistant** selects the points to evenly span the output variable *y*
- **Kennard Stone** selects the points which are evenly distributed in feature space
- **Define** the user defines which points are selected

After that, the features need to be scaled to make sure that the descriptor set is on the same scale. This allows us to compare the sign and magnitude of the descriptor coefficients in the model equation.

To keep things simple, we will use a random train/test split method using the one implemented in `sklearn` (here we are holding back 20% of the data in the test set using `test_size=0.2`):

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

We can easily visualize the results of out train:test split using a histogram:

In [ ]:
sns.histplot(y_train, color="blue", label="Train", alpha=0.5)
sns.histplot(y_test, color="orange", label="Test", alpha=0.5)
plt.xlabel(response_col)
plt.legend()
plt.show()

Now we will scale the data using the `StandardScaler` from scikit-learn.

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

## **5.2 Generate the models**

Now, finally, we are ready to do some modeling! To run the forward stepwise search algorithm, we will use a package called `MLXTEND` (https://rasbt.github.io/mlxtend/) to make like a bit easier!

This is probably not in the environment, so you'll need to install it:

In [ ]:
! pip install mlxtend
# ! uv pip install mlxtend  # if using a uv virtual environment

In [ ]:
from mlxtend.feature_selection import SequentialFeatureSelector
from mlxtend.plotting import plot_sequential_feature_selection as plot_sfs

sfs = SequentialFeatureSelector(linear_model.LinearRegression(),
                                k_features=4,
                                forward=True,
                                floating=False,
                                scoring='r2',
                                verbose=2,
                                cv=None)
selected_features = sfs.fit(X_train_scaled, y_train)

Once this has completed, we can view the results which is what happens to the model error when we increase the number of features. It will list what features are used in each model generated.

In [ ]:
fig = plot_sfs(sfs.get_metric_dict(), kind='std_err')

In [ ]:
selected_feature_names = [X_names[idx] for idx in selected_features.k_feature_idx_]
print(f"Selected features: {selected_feature_names}")

In [ ]:
# list all of the models with their features and R^2 scores
print("Model\tFeatures\tR^2")
for i in sfs.subsets_.keys():
    features = sfs.subsets_[i]['feature_idx']
    r2_score = sfs.subsets_[i]['avg_score']
    feature_names = [X_names[idx] for idx in features]
    print(f"{i}\t{feature_names}\t{r2_score:.3f}")

To finish off, we will plot the final model to see what it looks like. This will bring back our test set to see how well our model performs when predicting an unseen sample.

In [ ]:
mlr = linear_model.LinearRegression()

mlr.fit(X_train_scaled[:, selected_features.k_feature_idx_], y_train)
y_train_pred = mlr.predict(X_train_scaled[:, selected_features.k_feature_idx_])
y_test_pred = mlr.predict(X_test_scaled[:, selected_features.k_feature_idx_])

In [ ]:
plot_model.plot_model(y_train, y_test, y_train_pred, y_test_pred)

If there is time, these two cells will run two cross-validation tests on the model - a leave-one-out and k-fold cross-validation.

In [ ]:
plot_model.run_loocv(mlr, X_train_scaled[:, selected_features.k_feature_idx_], y_train, y_train_pred)

In [ ]:
plot_model.run_foldcv(mlr, X_train_scaled[:, selected_features.k_feature_idx_], y_train, y_train_pred, n_splits=2)